In [1]:
import sys

dir = '../..'
if dir not in sys.path:
    sys.path.append(dir)

In [2]:
from immGen import *

### Verify the SelectType Module
we will check that is able to recognize all instruction types

In [3]:
hw = py4hw.HWSystem()

opcode = hw.wire('opcode', 7)
imm_type = hw.wire('imm_type', 3)

sel = SelectType(hw, 'selectType', opcode, imm_type)

In [4]:
import punxa
from punxa.assembly import assemble

In [5]:
test_opcode = assemble('addi x10, x11, 1') & ((1<<7)-1)
opcode.put(test_opcode)
hw.getSimulator().clk()
print('Type:', imm_type.get())


Type: 0


In [6]:
def check_type(ins, expected):
    test_opcode = assemble(ins) & ((1<<7)-1)
    opcode.put(test_opcode)
    hw.getSimulator().clk()
    val = imm_type.get()
    val_map = ['I', 'S', 'B', 'U', 'J']
    print('Instruction:', ins, '\tType:', val_map[val], '\tExpected:', val_map[expected], '\t[OK]' if (val == expected) else '\t[ERROR]')

In [7]:
check_type('addi x10, x11, 1', 0)
check_type('sw x10, 8(x15)', 1)
check_type('beq x5, x6, 2', 2)
check_type('lui x10, 0x12345', 3)
check_type('jal x0, 55', 4)

Instruction: addi x10, x11, 1 	Type: I 	Expected: I 	[OK]
Instruction: sw x10, 8(x15) 	Type: S 	Expected: S 	[OK]
Instruction: beq x5, x6, 2 	Type: B 	Expected: B 	[OK]
Instruction: lui x10, 0x12345 	Type: U 	Expected: U 	[OK]
Instruction: jal x0, 55 	Type: J 	Expected: J 	[OK]


### Verify the immGen Module 

In [8]:
hw = py4hw.HWSystem()

ir = hw.wire('ir', 32)
imm = hw.wire('imm', 32)

sel = immGen(hw, 'selectType', ir, imm)

In [27]:
def check_imm(ins, expected):
    test_ins = assemble(ins) 
    ir.put(test_ins)
    hw.getSimulator().clk()
    val = py4hw.IntegerHelper.c2_to_signed(imm.get(), 32)
    
    print('Instruction:', ins, '\tImm:', val, '\tExpected:', expected, '\t[OK]' if (val == expected) else '\t[ERROR]')

In [43]:
check_imm('addi x10, x11, 10', 10)
check_imm('addi x10, x11, -1', -1)
check_imm('sw x10, 8(x15)', 8)
check_imm('sw x10, -8(x15)', 8)
check_imm('beq x5, x6, 20', 20)
check_imm('beq x5, x6, -20', -20)
check_imm('lui x10, 45', 45)
check_imm('jal x0, 50', 50)
check_imm('jal x0, -50', -50)

Instruction: addi x10, x11, 10 	Imm: 10 	Expected: 10 	[OK]
Instruction: addi x10, x11, -1 	Imm: -1 	Expected: -1 	[OK]
Instruction: sw x10, 8(x15) 	Imm: 1864 	Expected: 8 	[ERROR]
Instruction: sw x10, -8(x15) 	Imm: -168 	Expected: 8 	[ERROR]
Instruction: beq x5, x6, 20 	Imm: 20 	Expected: 20 	[OK]
Instruction: beq x5, x6, -20 	Imm: -20 	Expected: -20 	[OK]
Instruction: lui x10, 45 	Imm: 184320 	Expected: 45 	[ERROR]
Instruction: jal x0, 50 	Imm: 50 	Expected: 50 	[OK]
Instruction: jal x0, -50 	Imm: -50 	Expected: -50 	[OK]
